In [ ]:
import pandas as pd
import re
from pathlib import Path

# ============================================================
# PATH
# ============================================================

OUTPUT_DIR = Path(
    r"..\Output"
)

# Existing file containing 305 matched villages
DESA_FILE = OUTPUT_DIR / "Desa.csv"

# Centroid file containing BIG village polygons/centroids
CENTROID_FILE = Path(
    r"..\Output\Desa_Centroid_Master.csv"
)

# New revised output
OUTPUT_FILE = OUTPUT_DIR / "Desa_Revised.csv"


# ============================================================
# 1. NORMALIZATION FUNCTION
# ============================================================

def normalize_text(value):
    """
    Normalize text for matching village names.

    Example:
    AKAT FADEDO
    AKATFADEDO

    Both become:
    AKATFADEDO
    """

    if pd.isna(value):
        return ""

    return re.sub(
        r"[^A-Z0-9]",
        "",
        str(value).upper()
    )


# ============================================================
# 2. READ EXISTING DESA.CSV
# ============================================================

print("=" * 70)
print("MEMBACA DATA DESA.CSV")
print("=" * 70)

desa = pd.read_csv(
    DESA_FILE,
    dtype=str
)

print(f"Jumlah desa awal: {len(desa)}")


# ============================================================
# 3. READ CENTROID MASTER
# ============================================================

print("\nMembaca centroid master...")

centroid = pd.read_csv(
    CENTROID_FILE,
    dtype=str
)

print(f"Jumlah data centroid: {len(centroid)}")

print("\nKolom centroid:")
print(centroid.columns.tolist())


# ============================================================
# 4. CONVERT COORDINATES
# ============================================================

centroid["Longitude"] = pd.to_numeric(
    centroid["Longitude"],
    errors="coerce"
)

centroid["Latitude"] = pd.to_numeric(
    centroid["Latitude"],
    errors="coerce"
)


# ============================================================
# 5. TARGET VILLAGES TO ADD
# ============================================================

target_villages = pd.DataFrame({

    "Provinsi": [
        "MALUKU",
        "MALUKU",
        "MALUKU",
        "MALUKU"
    ],

    "Kabupaten/Kota": [
        "KAB. SERAM BAGIAN TIMUR",
        "KAB. SERAM BAGIAN TIMUR",
        "KAB. SERAM BAGIAN TIMUR",
        "KAB. SERAM BAGIAN TIMUR"
    ],

    "Kecamatan": [
        "UKAR SENGAN",
        "UKAR SENGAN",
        "UKAR SENGAN",
        "UKAR SENGAN"
    ],

    "Desa/Kelurahan": [
        "URUNG",
        "KWAWOR KECIL MATA ATA",
        "KWAWOR BESAR ENA",
        "AKAT FADEDO"
    ],

    "Kode Desa": [
        "8105162001",
        "8105162002",
        "8105162004",
        "8105162010"
    ]

})


# ============================================================
# 6. SEARCH CENTROID BY NORMALIZED VILLAGE NAME
# ============================================================

print("\n" + "=" * 70)
print("MENCARI 4 DESA DI CENTROID MASTER")
print("=" * 70)

# Create normalized village name
centroid["desa_name_key"] = centroid["NMDESA2"].apply(
    normalize_text
)

target_villages["desa_name_key"] = target_villages[
    "Desa/Kelurahan"
].apply(
    normalize_text
)


matched_rows = []
unmatched_rows = []

for _, target in target_villages.iterrows():

    matches = centroid[
        centroid["desa_name_key"]
        == target["desa_name_key"]
    ]

    # --------------------------------------------------------
    # Exact normalized name match
    # --------------------------------------------------------

    if len(matches) == 1:

        match = matches.iloc[0]

        print(
            f"FOUND: {target['Desa/Kelurahan']} "
            f"-> {match['NMDESA2']}"
        )

        new_row = {

            "KDDESA": target["Kode Desa"],

            "NMDESA2": target["Desa/Kelurahan"],

            "NMKEC": target["Kecamatan"],

            "NMDATI2": target["Kabupaten/Kota"],

            "NMPROP": target["Provinsi"],

            "Longitude": match["Longitude"],

            "Latitude": match["Latitude"]

        }

        matched_rows.append(new_row)

    elif len(matches) == 0:

        print(
            f"NOT FOUND: "
            f"{target['Desa/Kelurahan']}"
        )

        unmatched_rows.append(
            target["Desa/Kelurahan"]
        )

    else:

        print(
            f"AMBIGUOUS: "
            f"{target['Desa/Kelurahan']} "
            f"({len(matches)} matches)"
        )


# ============================================================
# 7. ADD BATU KARANG MANUALLY
# ============================================================

print("\n" + "=" * 70)
print("MENAMBAHKAN BATU KARANG SECARA MANUAL")
print("=" * 70)

batu_karang = {

    "KDDESA": "8109062011",

    "NMDESA2": "BATU KARANG",

    "NMKEC": "FENA FAFAN",

    "NMDATI2": "KAB. BURU SELATAN",

    "NMPROP": "MALUKU",

    "Longitude": 126.68833797863786,

    "Latitude": -3.5045160167308684

}

matched_rows.append(batu_karang)

print(
    "BATU KARANG ditambahkan dengan koordinat manual:"
)

print(
    f"Longitude: {batu_karang['Longitude']}"
)

print(
    f"Latitude : {batu_karang['Latitude']}"
)


# ============================================================
# 8. CONVERT NEW DATA TO DATAFRAME
# ============================================================

new_villages = pd.DataFrame(
    matched_rows
)


# ============================================================
# 9. CHECK DUPLICATES
# ============================================================

print("\n" + "=" * 70)
print("CEK DUPLIKAT")
print("=" * 70)

existing_codes = set(
    desa["KDDESA"].astype(str)
)

new_villages = new_villages[
    ~new_villages["KDDESA"].astype(str).isin(
        existing_codes
    )
].copy()

print(
    f"Desa baru yang akan ditambahkan: "
    f"{len(new_villages)}"
)


# ============================================================
# 10. COMBINE ORIGINAL + NEW VILLAGES
# ============================================================

revised = pd.concat(
    [
        desa,
        new_villages
    ],
    ignore_index=True
)


# ============================================================
# 11. SAVE REVISED CSV
# ============================================================

revised.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("SELESAI")
print("=" * 70)

print(
    f"Desa awal       : {len(desa)}"
)

print(
    f"Desa ditambahkan: {len(new_villages)}"
)

print(
    f"Total akhir     : {len(revised)}"
)

print("\nDesa yang ditambahkan:")

print(
    new_villages[
        [
            "KDDESA",
            "NMDESA2",
            "NMKEC",
            "NMDATI2",
            "NMPROP",
            "Longitude",
            "Latitude"
        ]
    ].to_string(index=False)
)

print("\nOutput tersimpan di:")

print(OUTPUT_FILE)